# Solution 2.7: Transforming and Merging (Angola IEA and INE trade)

Two sources, two jobs. The survey cleaned in 2.6 becomes analysis ready through
custom functions and `apply`. The INE trade workbooks are then loaded, merged and
stacked.

**PT:** Duas fontes, dois trabalhos. O inquerito limpo em 2.6 torna-se pronto
para analise com funcoes proprias e `apply`. Depois carregamos, juntamos e
empilhamos os ficheiros de comercio do INE.

> **Pipeline:** run 2.6 first. Reads `10_cleaned/` and `0_raw/angola`, writes
> `20_processed/`.

### Path Setup (run first)

**PT:** Configuracao dos caminhos.

In [1]:
import os

import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_CLEAN_DIR = '../../data/10_cleaned'
DATA_PROC_DIR = '../../data/20_processed'

TRADE_DIR = 'international_trade'

clean_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')
trade_dir = os.path.join(DATA_RAW_DIR, TRADE_DIR)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Trade workbooks available / Ficheiros de comercio disponiveis:')
for name in sorted(os.listdir(trade_dir)):
    print('  ', name)

Trade workbooks available / Ficheiros de comercio disponiveis:
   Comercio Externo de Bens por Grandes Categorias Económicas.xlsx
   Comercio Externo de Bens por Países Parceiros.xlsx
   Comércio Externo de Bens por Categorias Produtos.xlsx
   Comércio Externo de Bens por Produtos.xlsx
   Comércio Externo de Bens por Reexportação e Reimportação.xlsx
   Comércio Externo de Bens por Secções e Capítulos.xlsx
   Comércio Externos de Bens por Categorias de Produtos.xlsx


---

# Part A: the survey

## Task 1: Load the survey and restore its types

CSV forgets dtypes. Notebook 2.6 saved them in the codebook, so read that file
first and use it.

**What to do:** read the codebook, build two dictionaries from it, one mapping
`new_name` to `description` and one mapping `new_name` to `dtype_final`, then
pass the dtypes to `read_csv`. Datetime columns cannot go through `dtype=`, so
they are separated into `parse_dates`.

**PT:** O CSV esquece os tipos. O caderno 2.6 gravou-os no dicionario.

**O que fazer:** leia o dicionario, construa dois dicionarios a partir dele, um
de `new_name` para `description` e outro de `new_name` para `dtype_final`, e
passe os tipos ao `read_csv`. As colunas de data nao podem ir em `dtype=`, por
isso vao em `parse_dates`.

In [2]:
codebook_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_codebook.csv')
codebook_df = pd.read_csv(codebook_path)

descriptions = dict(zip(codebook_df['new_name'], codebook_df['description']))
dtypes = dict(zip(codebook_df['new_name'], codebook_df['dtype_final']))

# dtype= cannot build a datetime, those columns need parse_dates instead
# dtype= nao constroi datas, essas colunas precisam de parse_dates
date_cols = [col for col, kind in dtypes.items() if kind.startswith('datetime')]
read_dtypes = {col: kind for col, kind in dtypes.items()
               if not kind.startswith('datetime')}

df = pd.read_csv(clean_path, dtype=read_dtypes, parse_dates=date_cols)

print('Survey:', df.shape)
print(df[['household_id', 'age', 'job_start_year', 'interview_date']].dtypes)

Survey: (53353, 27)
household_id      string[python]
age                        int64
job_start_year             Int64
interview_date    datetime64[ns]
dtype: object


**Answers:**

- 53,353 rows and 27 columns, with every type restored: `household_id` is text,
  `age` a plain integer, `job_start_year` a nullable integer, `interview_date` a
  datetime.
- Without the codebook every notebook downstream would redo the casting by hand
  and could quietly disagree about it.
- `interview_date` is the exception: `dtype=` only builds simple types, so
  datetimes go through `parse_dates`.

**PT:** 53.353 linhas e 27 colunas, com todos os tipos restaurados. Sem o
dicionario, cada caderno seguinte repetiria a conversao a mao. `interview_date` e
a excecao: `dtype=` so constroi tipos simples.

---

## Task 2: Write a function and apply it to one column

`apply` on a Series runs your function once per value. Use it when the rule needs
branching that a single expression cannot express clearly.

**What to do:** complete `age_band` so it returns `'Child'` under 15, `'Youth'`
under 25, `'Adult'` under 65 and `'Elderly'` otherwise, then apply it to `age` and
store the result in a new column `age_band`.

**PT:** `apply` numa Serie corre a funcao para cada valor.

**O que fazer:** complete `age_band` para devolver `'Child'` abaixo de 15,
`'Youth'` abaixo de 25, `'Adult'` abaixo de 65 e `'Elderly'` nos restantes casos.
Depois aplique a coluna `age` e guarde em `age_band`.

In [3]:
def age_band(age):
    """ILO oriented age band for one person / Faixa etaria para uma pessoa."""
    if age < 15:
        return 'Child'
    if age < 25:
        return 'Youth'
    if age < 65:
        return 'Adult'
    return 'Elderly'


df['age_band'] = df['age'].apply(age_band)
df['age_band'].value_counts()

age_band
Child      23528
Adult      17239
Youth      10901
Elderly     1685
Name: count, dtype: int64

**Answers:**

- Child 23,528, Youth 10,901, Adult 17,239, Elderly 1,685.
- The thresholds are not arbitrary: 15 is the ILO working age threshold and 65
  the usual retirement reference, so the function encodes a definition.
- No `pd.isna` guard is needed because 2.6 cast `age` to a plain integer, which
  cannot hold a missing value.

**PT:** Child 23.528, Youth 10.901, Adult 17.239, Elderly 1.685. Os limiares nao
sao arbitrarios: 15 anos e o limiar de idade ativa da OIT e 65 a referencia de
reforma. Nao e preciso proteger contra valores em falta porque 2.6 converteu
`age` para inteiro simples.

---

## Task 3: Write a function that reads several columns at once

`apply(axis=1)` passes a whole row to your function, so it can read many columns
together. Labour force status is the natural case: it depends on eight answers,
and no single column expression can express it.

Because 2.6 kept the value labels, the answers are the Portuguese words `Sim` and
`Nao`, so the function reads almost like the questionnaire.

**What to do:** complete `labour_force_status` following the ILO rule:

1. under 15 years old, return `'Outside labour force'`
2. answered `Sim` to any of `worked_for_pay`, `worked_own_account` or
   `absent_from_job`, return `'Employed'`
3. answered `Sim` to `sought_job` **or** `sought_business`, **and** `Sim` to
   `available_last_week` **or** `available_next_2weeks`, return `'Unemployed'`
4. otherwise return `'Outside labour force'`

Then apply it with `axis=1` and store the result in `lf_status`.

**PT:** `apply(axis=1)` passa a linha inteira, por isso a funcao pode ler varias
colunas. Como 2.6 manteve as etiquetas, as respostas sao `Sim` e `Nao`.

**O que fazer:** complete `labour_force_status` segundo a regra da OIT: menos de
15 anos, fora da forca de trabalho; `Sim` a qualquer uma das tres perguntas de
trabalho, empregado; `Sim` a procura **e** `Sim` a disponibilidade, desempregado;
caso contrario, fora da forca de trabalho. Aplique com `axis=1` e guarde em
`lf_status`.

In [4]:
def labour_force_status(row):
    """ILO status for one person / Situacao perante o trabalho de uma pessoa."""
    if row['age'] < 15:
        return 'Outside labour force'

    worked = (row['worked_for_pay'], row['worked_own_account'], row['absent_from_job'])
    if 'Sim' in worked:
        return 'Employed'

    searched = 'Sim' in (row['sought_job'], row['sought_business'])
    available = 'Sim' in (row['available_last_week'], row['available_next_2weeks'])
    if searched and available:
        return 'Unemployed'

    return 'Outside labour force'


df['lf_status'] = df.apply(labour_force_status, axis=1)
df['lf_status'].value_counts()

lf_status
Outside labour force    34975
Employed                15745
Unemployed               2633
Name: count, dtype: int64

**Answers:**

- 15,745 employed, 2,633 unemployed, 34,975 outside the labour force.
- Availability tests `available_last_week` **or** `available_next_2weeks` because
  the second is only asked of people who answered no to the first. Testing the
  second alone gives an unemployment rate of 0.7%, which looks like a bug but is
  really a misread skip pattern.
- `apply(axis=1)` walks 53,353 rows in Python and is slow. It earns that here
  because the rule is a definition somebody must be able to audit line by line.

**PT:** 15.745 empregados, 2.633 desempregados, 34.975 fora da forca de trabalho.
A disponibilidade testa as duas perguntas com **ou** porque a segunda so e feita a
quem respondeu nao a primeira. `apply(axis=1)` e lento, mas aqui compensa porque
a regra e uma definicao que alguem tem de poder auditar.

---

## Task 4: Weight the result

Each person represents many Angolans, and the `weight` column says how many. An
unweighted rate describes the sample; a weighted rate describes the country.

**What to do:** complete `weighted_share` so it returns the weighted percentage
of the population picked out by a boolean mask, then use it for the unemployment
rate, which is over the labour force, and the participation rate, which is over
the working age population.

**PT:** Cada pessoa representa muitos angolanos, e a coluna `weight` diz quantos.

**O que fazer:** complete `weighted_share` para devolver a percentagem ponderada
selecionada por uma mascara, e use-a para a taxa de desemprego, sobre a forca de
trabalho, e a taxa de atividade, sobre a populacao em idade ativa.

In [5]:
def weighted_share(mask, weights):
    """Weighted percentage selected by `mask` / Percentagem ponderada."""
    return weights[mask].sum() / weights.sum() * 100


weight = df['weight']
in_labour_force = df['lf_status'].isin(['Employed', 'Unemployed'])
working_age = df['age'] >= 15

unemployment = weighted_share(df['lf_status'] == 'Unemployed', weight[in_labour_force])
participation = weighted_share(in_labour_force, weight[working_age])

print(f'Unemployment rate:  {unemployment:5.1f}%')
print(f'Participation rate: {participation:5.1f}%')

Unemployment rate:   14.5%
Participation rate:  62.8%


**Answers:**

- Unemployment 14.5%, participation 62.8%.
- This is the strict definition, counting only people who actively searched. A
  relaxed measure that also counts those who want work but stopped looking runs
  far higher, and INE's published headline is closer to that, so always say which
  one a table reports.
- The weights matter most for totals. Here they barely move the rate, which is
  luck rather than a reason to skip them.

**PT:** Desemprego 14,5%, atividade 62,8%. Esta e a definicao estrita. Uma medida
alargada da valores bem mais altos, e o numero publicado pelo INE esta mais
proximo dessa, por isso diga sempre qual usou.

---

# Part B: the trade workbooks

## Task 5: Load two trade sheets and tidy their column names

INE publishes the trade data as spreadsheets made for human readers: two title
rows above the header, a blank row, a `Total Geral` row, the data, and a source
footer at the bottom.

**What to do:**

1. load the export sheet and the import sheet with `skiprows=2`, so the real
   header becomes the header, into `export_df` and `import_df`
2. print the column names of both and look at what arrived
3. convert those names to snake case, then keep only the rows where the country
   name is filled in, which drops the blank row, the total and the footer at once

**PT:** O INE publica os dados em folhas feitas para leitura humana: duas linhas
de titulo, o cabecalho, uma linha vazia, o `Total Geral`, os dados, e o rodape.

**O que fazer:** carregue as duas folhas com `skiprows=2` para `export_df` e
`import_df`; imprima os nomes das colunas; converta esses nomes para snake case e
mantenha so as linhas com o nome do pais preenchido.

In [6]:
PARTNERS_FILE = 'Comercio Externo de Bens por Países Parceiros.xlsx'
partners_path = os.path.join(trade_dir, PARTNERS_FILE)

export_df = pd.read_excel(partners_path, sheet_name='Exportação por Países (USD)',
                          skiprows=2)
import_df = pd.read_excel(partners_path, sheet_name='Importação por Países (USD)',
                          skiprows=2)

print('export_df:', export_df.shape, '| import_df:', import_df.shape)

export_df: (252, 24) | import_df: (252, 24)


In [7]:
# Look at the names before touching them / Ver os nomes antes de mexer
print(list(export_df.columns))

['Código', 'País', 'Ano\n2004', 'Ano\n2005', 'Ano\n2006', 'Ano\n2007', 'Ano\n2008', 'Ano\n2009', 'Ano\n2010', 'Ano\n2011', 'Ano\n2012', 'Ano\n2013', 'Ano\n2014', 'Ano\n2015', 'Ano\n2016', 'Ano\n2017', 'Ano\n2018', 'Ano\n2019', 'Ano\n2020', 'Ano\n2021', 'Ano\n2022', 'Ano\n2023', 'Ano\n2024', 'Ano\n2025']


In [8]:
def to_snake_case(columns):
    """Lower case, strip accents, and join words with underscores.

    Minusculas, sem acentos, e palavras unidas por underscore.
    """
    return (columns
            .str.replace('\n', ' ', regex=False)
            .str.strip()
            .str.lower()
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.replace(' ', '_', regex=False))


export_df.columns = to_snake_case(export_df.columns)
import_df.columns = to_snake_case(import_df.columns)

# Rows without a country name are the blank row, the total and the footer
# As linhas sem nome de pais sao a linha vazia, o total e o rodape
export_df = export_df[export_df['pais'].notna()]
import_df = import_df[import_df['pais'].notna()]

print(list(export_df.columns)[:4], '...', list(export_df.columns)[-1:])
print('export_df:', export_df.shape, '| import_df:', import_df.shape)
export_df[['codigo', 'pais', 'ano_2025']].head()

['codigo', 'pais', 'ano_2004', 'ano_2005'] ... ['ano_2025']
export_df: (249, 24) | import_df: (249, 24)


,codigo,pais,ano_2025
2,AF,Afeganistão,0.41
3,ZA,África do Sul,"593,519.94"
4,AL,Albânia,2.52
5,DE,Alemanha,"55,864.20"
6,AD,Andorra,3.90


**Answers:**

- The raw names are `Código`, `País` and `Ano\n2004` onwards. The year columns
  carry a line break inside the name, invisible when printed and fatal when you
  try to reference one.
- After tidying they are `codigo`, `pais`, `ano_2004` to `ano_2025`: lower case,
  no accents, no spaces, so they can be typed without surprises.
- 249 rows each: 248 partner countries plus `ZZ`, Desconhecido, meaning the
  partner was not recorded.
- The values are in **thousands of US dollars**, stated only in the sheet's second
  line. No column name says so, which is how unit errors reach publication.

**PT:** Os nomes originais tem acentos e uma quebra de linha invisivel. Depois de
arrumados ficam `codigo`, `pais`, `ano_2004` a `ano_2025`. Sao 249 linhas: 248
paises mais `ZZ`, Desconhecido. Os valores estao em **milhares de dolares**,
indicado apenas na segunda linha da folha.

---

## Task 6: Merge the two flows and classify each partner

Both tables have one row per country, so this is a one to one merge.

**What to do:**

1. merge `export_df` and `import_df` on `codigo`, keeping both sides, with
   `indicator=True` and `validate='one_to_one'`, renaming the two value columns
   to `exports_thousand_usd` and `imports_thousand_usd`, and `codigo` and `pais`
   to `country_code` and `country_name`
2. compute `balance_thousand_usd` as exports minus imports, treating a missing
   flow as zero
3. write `partner_profile`, which needs both value columns at once and therefore
   runs with `axis=1`

**PT:** As duas tabelas tem uma linha por pais, por isso a juncao e um para um.

**O que fazer:** junte `export_df` e `import_df` por `codigo` com
`indicator=True` e `validate='one_to_one'`; calcule `balance_thousand_usd`; e
escreva `partner_profile`, que precisa das duas colunas ao mesmo tempo e corre
com `axis=1`.

In [9]:
YEAR = 'ano_2025'

trade = pd.merge(
    export_df[['codigo', 'pais', YEAR]].rename(columns={YEAR: 'exports_thousand_usd'}),
    import_df[['codigo', YEAR]].rename(columns={YEAR: 'imports_thousand_usd'}),
    on='codigo', how='outer', indicator=True, validate='one_to_one',
).rename(columns={'codigo': 'country_code', 'pais': 'country_name'})

print(trade['_merge'].value_counts())
trade = trade.drop(columns='_merge')
print('Merged:', trade.shape)

_merge
both          249
left_only       0
right_only      0
Name: count, dtype: int64
Merged: (249, 4)


In [10]:
def partner_profile(row):
    """Describe Angola's 2025 relationship with one partner.

    Return, in this order of priority:
      'Incomplete'          when either flow is missing
      'Negligible'          when the two flows together are under 1000
      'Angola mainly sells' when exports are more than double imports
      'Angola mainly buys'  when imports are more than double exports
      'Two way'             otherwise

    Devolver, por esta ordem de prioridade: 'Incomplete' se faltar um fluxo,
    'Negligible' se a soma for inferior a 1000, 'Angola mainly sells' se as
    exportacoes forem mais do dobro das importacoes, 'Angola mainly buys' no caso
    inverso, e 'Two way' nos restantes casos.
    """
    sold = row['exports_thousand_usd']
    bought = row['imports_thousand_usd']

    if pd.isna(sold) or pd.isna(bought):
        return 'Incomplete'
    if sold + bought < 1000:
        return 'Negligible'
    if sold > 2 * bought:
        return 'Angola mainly sells'
    if bought > 2 * sold:
        return 'Angola mainly buys'
    return 'Two way'


trade['balance_thousand_usd'] = (trade['exports_thousand_usd'].fillna(0)
                                 - trade['imports_thousand_usd'].fillna(0))
trade['profile'] = trade.apply(partner_profile, axis=1)

print(trade['profile'].value_counts())

profile
Negligible             141
Angola mainly buys      68
Angola mainly sells     22
Two way                 18
Name: count, dtype: int64


In [11]:
print('Largest surpluses / Maiores excedentes:')
print(trade.nlargest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))
print()
print('Largest deficits / Maiores defices:')
print(trade.nsmallest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))

Largest surpluses / Maiores excedentes:
          country_name  exports_thousand_usd  imports_thousand_usd             profile
                 China         14,450,839.72          3,420,626.27 Angola mainly sells
                 Índia          3,639,895.11          1,043,808.31 Angola mainly sells
             Indonésia          1,967,048.85            146,711.05 Angola mainly sells
               Espanha          1,721,616.77            284,839.90 Angola mainly sells
Emirados Árabes Unidos          1,722,331.20            683,052.59 Angola mainly sells

Largest deficits / Maiores defices:
             country_name  exports_thousand_usd  imports_thousand_usd            profile
                 Portugal            169,020.79          1,642,901.16 Angola mainly buys
              Reino Unido            172,270.94          1,002,682.20 Angola mainly buys
            Coreia do Sul                 70.40            805,860.19 Angola mainly buys
                Argentina                188.

**Answers:**

- All 249 partners are `both`, and `validate='one_to_one'` passed, so each appears
  exactly once on each side.
- 141 partners are negligible, 68 mainly sell to Angola, 22 mainly buy from it,
  18 are two way.
- China is by far the largest surplus partner and Portugal the largest deficit,
  which matches Angola's oil export and consumer goods import profile.
- The `Negligible` threshold is a judgement, not a fact. Without it the ratio
  rules would label a partner with 3 thousand dollars of trade as "mainly sells",
  which is technically true and useless.

**PT:** Todos os 249 parceiros correspondem e a validacao passou. A China e o
maior excedente e Portugal o maior defice. O limiar `Negligible` e um julgamento,
nao um facto.

---

## Task 7: Do not sum a hierarchy by accident

This second workbook breaks exports down by economic category. Its code column
holds a tree: a one digit code is a section, two digits a group inside that
section, three digits a subgroup inside that group.

So the rows are not comparable. Adding them all up counts the same money more
than once. The sheet publishes its own `Total Geral`, which is the check.

**What to do:** load and tidy this sheet the same way as before, look at the
codes, set the published total aside, then write `cgce_level` to measure how deep
each code sits and use it to find which rows may be summed.

**PT:** Este segundo ficheiro reparte as exportacoes por categoria economica. A
coluna de codigo guarda uma arvore: um digito e uma seccao, dois digitos um grupo
dentro dela, tres digitos um subgrupo. As linhas nao sao comparaveis, e soma-las
todas conta o mesmo dinheiro mais do que uma vez.

**O que fazer:** carregue e arrume esta folha como antes, veja os codigos, guarde
o total publicado, e escreva `cgce_level` para medir a profundidade de cada
codigo.

In [12]:
CGCE_FILE = 'Comercio Externo de Bens por Grandes Categorias Económicas.xlsx'
cgce_path = os.path.join(trade_dir, CGCE_FILE)

cgce_df = pd.read_excel(cgce_path, sheet_name='Export Cat. Económica (USD)',
                        skiprows=2)
cgce_df.columns = to_snake_case(cgce_df.columns)
cgce_df = cgce_df[cgce_df['descricao'].notna()]

print('cgce_df:', cgce_df.shape)
print(list(cgce_df.columns)[:4])

cgce_df: (28, 24)
['cgce', 'descricao', 'ano_2004', 'ano_2005']


In [13]:
# The codes get longer as the categories get narrower
# Os codigos ficam mais longos a medida que as categorias se estreitam
cgce_df[['cgce', 'descricao', YEAR]].head(8)

,cgce,descricao,ano_2025
1,NaN,Total Geral,"30,727,896.04"
2,1,Alimentos e bebidas,"158,555.02"
3,11,"Alimentos e bebidas, primário","24,741.63"
4,111,"Alimentos e bebidas, primários, principalmente...","10,416.96"
5,112,"Alimentos e bebidas, primárias, principalmente...","14,305.99"
6,12,Alimentos e bebidas transformados,"133,813.39"
7,121,"Alimentos e bebidas, transformados, principalm...","64,576.99"
8,122,"Alimentos e bebidas, transformados, principalm...","32,492.84"


In [14]:
# Keep the published total aside, then drop that row from the data
# Guardar o total publicado e remover essa linha dos dados
published_total = cgce_df.loc[cgce_df['descricao'] == 'Total Geral', YEAR].iloc[0]
cgce_df = cgce_df[cgce_df['cgce'].notna()]

print('Published Total Geral:', f'{published_total:,.0f}')
print('Category rows:', len(cgce_df))

Published Total Geral: 30,727,896
Category rows: 27


In [15]:
def cgce_level(code):
    """How deep a code sits: 1 section, 2 group, 3 subgroup.

    Profundidade do codigo: 1 seccao, 2 grupo, 3 subgrupo.
    """
    return len(str(code).strip())


cgce_df['level'] = cgce_df['cgce'].apply(cgce_level)
print(cgce_df['level'].value_counts().sort_index())

level
1     7
2    14
3     6
Name: count, dtype: int64


In [16]:
naive = cgce_df[YEAR].sum()
sections_only = cgce_df.loc[cgce_df['level'] == 1, YEAR].sum()

print(f'Published total:     {published_total:15,.0f}')
print(f'Sum of every row:    {naive:15,.0f}  <- {naive / published_total:.2f}x')
print(f'Sum of level 1 only: {sections_only:15,.0f}')
print()
print('Level 1 matches the published total:',
      bool(abs(sections_only - published_total) < 1))

Published total:          30,727,896
Sum of every row:         61,593,267  <- 2.00x
Sum of level 1 only:      30,727,896

Level 1 matches the published total: True


**Answers:**

- 7 sections, 14 groups, 6 subgroups.
- Summing every row gives **exactly twice** the published total, because each
  section already contains its groups and each group its subgroups, so every
  value is counted once at its own level and again in every ancestor.
- Summing only level 1 reproduces the published `Total Geral` to the unit, which
  is how you know the rule is right rather than merely plausible.
- Nothing warned you. No error is raised, and a chart built on the naive sum is
  silently double the truth.

**PT:** 7 seccoes, 14 grupos, 6 subgrupos. Somar todas as linhas da exatamente o
dobro do total publicado. Somar apenas o nivel 1 reproduz o `Total Geral`
publicado ao cêntimo. Nada avisa: nenhum erro e levantado.

---

## Task 8: Save

**What to do:** write the survey and the trade table to `20_processed/` with
`index=False`.

**PT:** **O que fazer:** grave o inquerito e a tabela de comercio em
`20_processed/` com `index=False`.

In [17]:
os.makedirs(DATA_PROC_DIR, exist_ok=True)

survey_out = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_features.csv')
trade_out = os.path.join(DATA_PROC_DIR, 'angola_trade_partners.csv')

df.to_csv(survey_out, index=False)
trade.to_csv(trade_out, index=False)

print('survey:', df.shape, '| trade:', trade.shape)

survey: (53353, 29) | trade: (249, 6)


**Answers:**

- The survey gained `age_band` and `lf_status`, both produced by functions
  somebody can read and argue with, which is the point when a number ends up in a
  publication.
- Nothing was written into `0_raw/`.

**PT:** O inquerito ganhou `age_band` e `lf_status`, ambos produzidos por funcoes
que alguem pode ler e contestar. Nada foi escrito em `0_raw/`.